# Librerías

In [1]:
import os
import re
import time
import json
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from collections import namedtuple
from datetime import date

# Lista de URLs

In [2]:
base = "https://inmuebles.mercadolibre.com.mx"

tipos = ["casas", "departamentos"]
operaciones = ["venta", "renta"]

# -------------------------
# 1) CDMX: 16 alcaldías
# -------------------------
cdmx_alcaldias = [
    "alvaro-obregon",
    "azcapotzalco",
    "benito-juarez",
    "coyoacan",
    "cuajimalpa-de-morelos",
    "cuauhtemoc",
    "gustavo-a-madero",
    "iztacalco",
    "iztapalapa",
    "la-magdalena-contreras",
    "miguel-hidalgo",
    "milpa-alta",
    "tlahuac",
    "tlalpan",
    "venustiano-carranza",
    "xochimilco",
]

# -------------------------
# 2) EdoMex metropolitano
# -------------------------
edomex_metro = [
    "atizapan-de-zaragoza",
    "coacalco-de-berriozabal",
    "cuautitlan",
    "cuautitlan-izcalli",
    "chalco",
    "chicoloapan",
    "chimalhuacan",
    "ecatepec-de-morelos",
    "huixquilucan",
    "ixtapaluca",
    "la-paz",
    "naucalpan-de-juarez",
    "nezahualcoyotl",
    "nicolas-romero",
    "tecamac",
    "teoloyucan",
    "tepotzotlan",
    "texcoco",
    "tlalnepantla-de-baz",
    "tultitlan",
    "valle-de-chalco-solidaridad",
    "zumpango",
]

# -------------------------
# 3) Estados a nivel general
# -------------------------
estados_generales = [
    "aguascalientes",
    "baja-california",
    "baja-california-sur",
    "campeche",
    "chiapas",
    "chihuahua",
    "coahuila",
    "colima",
    "durango",
    "guanajuato",
    "guerrero",
    "hidalgo",
    "jalisco",
    "michoacan",
    "morelos",
    "nayarit",
    "nuevo-leon",
    "oaxaca",
    "puebla",
    "queretaro",
    "quintana-roo",
    "san-luis-potosi",
    "sinaloa",
    "sonora",
    "tabasco",
    "tamaulipas",
    "tlaxcala",
    "veracruz",
    "yucatan",
    "zacatecas",
]

# -------------------------
# 4) Ciudades / municipios grandes
# -------------------------
ciudades_grandes = {
    "aguascalientes": ["aguascalientes", "jesus-maria"],
    "baja-california": ["tijuana", "mexicali", "ensenada", "tecate", "playas-de-rosarito"],
    "baja-california-sur": ["la-paz", "los-cabos"],
    "campeche": ["campeche", "carmen"],
    "chiapas": ["tuxtla-gutierrez", "san-cristobal-de-las-casas", "tapachula"],
    "chihuahua": ["chihuahua", "juarez", "delicias"],
    "coahuila": ["saltillo", "torreon", "monclova", "piedras-negras", "ramos-arizpe"],
    "colima": ["colima", "manzanillo", "villa-de-alvarez"],
    "durango": ["durango", "gomez-palacio", "lerdo"],
    "guanajuato": ["leon", "celaya", "irapuato", "salamanca", "guanajuato", "san-miguel-de-allende"],
    "guerrero": ["acapulco-de-juarez", "chilpancingo-de-los-bravo", "iguala-de-la-independencia"],
    "hidalgo": ["pachuca-de-soto", "mineral-de-la-reforma", "tizayuca", "tula-de-allende"],
    "jalisco": ["guadalajara", "zapopan", "tlajomulco-de-zuniga", "san-pedro-tlaquepaque", "tonala", "puerto-vallarta"],
    "michoacan": ["morelia", "uruapan", "zamora", "lazaro-cardenas"],
    "morelos": ["cuernavaca", "jiutepec", "temixco", "yautepec", "emiliano-zapata", "xochitepec", "cuautla"],
    "nayarit": ["tepic", "bahia-de-banderas"],
    "nuevo-leon": ["monterrey", "guadalupe", "san-nicolas-de-los-garza", "apodaca", "general-escobedo", "santa-catarina", "san-pedro-garza-garcia", "garcia", "juarez"],
    "oaxaca": ["oaxaca-de-juarez", "santa-cruz-xoxocotlan", "san-juan-bautista-tuxtepec"],
    "puebla": ["puebla", "san-andres-cholula", "san-pedro-cholula", "cuautlancingo", "tehuacan", "coronango", "atlixco"],
    "queretaro": ["queretaro", "el-marques", "corregidora", "san-juan-del-rio"],
    "quintana-roo": ["benito-juarez", "solidaridad", "tulum", "othon-p-blanco", "isla-mujeres", "puerto-morelos"],
    "san-luis-potosi": ["san-luis-potosi", "soledad-de-graciano-sanchez"],
    "sinaloa": ["culiacan", "mazatlan", "ahome", "guasave"],
    "sonora": ["hermosillo", "cajeme", "nogales", "san-luis-rio-colorado"],
    "tabasco": ["centro", "paraiso", "comalcalco"],
    "tamaulipas": ["reynosa", "matamoros", "nuevo-laredo", "tampico", "ciudad-madero", "altamira", "victoria"],
    "tlaxcala": ["tlaxcala", "apizaco", "chiautempan", "huamantla"],
    "veracruz": ["veracruz", "boca-del-rio", "xalapa", "coatzacoalcos", "cordoba", "orizaba", "poza-rica-de-hidalgo"],
    "yucatan": ["merida", "progreso", "valladolid"],
    "zacatecas": ["zacatecas", "guadalupe", "fresnillo"],
}

# -------------------------
# Generación de URLs
# -------------------------
urls = []

# CDMX
urls += [
    f"{base}/{tipo}/{operacion}/distrito-federal/{alcaldia}/"
    for alcaldia in cdmx_alcaldias
    for tipo in tipos
    for operacion in operaciones
]

# EdoMex metropolitano
urls += [
    f"{base}/{tipo}/{operacion}/estado-de-mexico/{municipio}/"
    for municipio in edomex_metro
    for tipo in tipos
    for operacion in operaciones
]

# Estados generales
urls += [
    f"{base}/{tipo}/{operacion}/{estado}/"
    for estado in estados_generales
    for tipo in tipos
    for operacion in operaciones
]

# Ciudades / municipios grandes
urls += [
    f"{base}/{tipo}/{operacion}/{estado}/{ciudad}/"
    for estado, ciudades in ciudades_grandes.items()
    for ciudad in ciudades
    for tipo in tipos
    for operacion in operaciones
]

# Quitar duplicados
urls = list(dict.fromkeys(urls))

print(f"Total URLs: {len(urls)}")

Total URLs: 776


# Funciones

In [3]:
ruta_archivo = "ws_vivienda.csv"
ruta_checkpoint = "checkpoint.json"

def guardar_dataframe(df, ruta):
    """Guarda (o agrega) el dataframe en CSV, deduplicando por link."""
    if os.path.exists(ruta):
        df_existente = pd.read_csv(ruta)
        df_combinado = pd.concat([df_existente, df], ignore_index=True)
        df_combinado.drop_duplicates(subset=['link'], inplace=True)
        df_combinado.to_csv(ruta, index=False)
    else:
        df.to_csv(ruta, index=False)

In [4]:
def extraer_datos_url(url):
    """Extrae tipo_vivienda, operacion, estado y municipio de la URL."""
    # Caso con estado y municipio
    m1 = re.search(r'/(casas|departamentos)/(venta|renta)/([^/]+)/([^/]+)/', url)
    if m1:
        tipo_vivienda, operacion, estado, municipio = m1.groups()
        return tipo_vivienda, operacion, estado, municipio

    # Caso con solo estado
    m2 = re.search(r'/(casas|departamentos)/(venta|renta)/([^/]+)/', url)
    if m2:
        tipo_vivienda, operacion, estado = m2.groups()
        return tipo_vivienda, operacion, estado, None

    return None, None, None, None

## Funciones Principales

In [5]:
Articulo = namedtuple("Articulo", ["fecha", "titulo", "recamaras", "banos", "superficie_m2", "precio_actual", "link", "imagen"])

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-MX,es;q=0.9"
}


def limpia_precio(txt):
    if not txt:
        return None
    t = txt.replace("\xa0", " ").replace(".", "").replace(",", "").replace("$", "").strip()
    try:
        return int("".join(ch for ch in t if ch.isdigit()))
    except ValueError:
        return None


def parse_item(li):
    """Parsea un elemento <li> de la lista de resultados de ML."""
    titulo = None
    link = None
    a = li.select_one("h3.poly-component__title-wrapper a.poly-component__title[href]")
    if a:
        titulo = a.get_text(strip=True)
        link = a["href"]

    # Imagen
    img = li.find("img")
    imagen = img.get("data-src") or img.get("src") if img else None

    # Precio actual
    precio_actual = None
    actual_span = li.select_one(".poly-price__current .andes-money-amount__fraction")
    if actual_span:
        precio_actual = limpia_precio(actual_span.get_text())

    # Atributos: recámaras, baños, superficie
    recamaras = None
    banos = None
    superficie_m2 = None

    attrs = li.select("ul.poly-attributes_list li.poly-attributes_list__item")
    for it in attrs:
        txt = it.get_text(" ", strip=True).lower()

        m = re.search(r"(\d+)\s*(recámaras|recamaras|habitaciones|dormitorios)", txt)
        if m:
            recamaras = int(m.group(1))
            continue

        m = re.search(r"(\d+)\s*(baños|banos)", txt)
        if m:
            banos = int(m.group(1))
            continue

        m = re.search(r"(\d+(?:\.\d+)?)\s*m²", txt)
        if m:
            superficie_m2 = float(m.group(1))
            continue

    return Articulo(
        date.today(),
        titulo,
        recamaras,
        banos,
        superficie_m2,
        precio_actual,
        link,
        imagen
    )

## Función de coordenadas

In [6]:
def get_lat_lon(url, headers=HEADERS):
    """Extrae latitud y longitud del HTML del anuncio individual."""
    url = url.split("?")[0].split("#")[0]
    try:
        resp = requests.get(url, headers=headers, timeout=25)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] Error al obtener coordenadas: {e}")
        return None, None

    m = re.search(
        r'"map_info".*?"location":\{"latitude":"?([-0-9.]+)"?,"longitude":"?([-0-9.]+)"?\}',
        resp.text,
        flags=re.DOTALL
    )

    if m:
        return float(m.group(1)), float(m.group(2))

    print("[INFO] No se encontraron coordenadas")
    return None, None

## Función de descripción

In [7]:
def get_descripcion(url, headers=HEADERS):
    """
    Devuelve la descripción completa del anuncio de MercadoLibre.
    Retorna None si no se encuentra.
    """
    try:
        resp = requests.get(url, headers=headers, timeout=25)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] Error al obtener la descripción: {e}")
        return None

    soup = BeautifulSoup(resp.text, "html.parser")

    desc_node = soup.select_one("p.ui-pdp-description__content")
    if not desc_node:
        desc_node = soup.select_one("[data-testid='content']")
    if not desc_node:
        print("[INFO] No se encontró la descripción en la página.")
        return None

    time.sleep(random.randint(30, 60))
    return desc_node.get_text(separator=" ", strip=True)

## Función de Scraping

In [8]:
ESPERA_S = 30  # Pausa entre páginas

def scrap_page(paginas, url, headers):
    """
    Scrapea múltiples páginas de resultados de una URL de ML.
    Retorna un DataFrame con todos los artículos encontrados.
    """
    articulos = []

    for page in range(1, paginas + 1):
        print(f"Scrapeando página {page}")
        sep = "" if url.endswith("/") else "/"
        url_pag = url if page == 1 else f"{url}{sep}_Desde_{(page - 1) * 48 + 1}"

        try:
            resp = requests.get(url_pag, headers=headers, timeout=20)
            resp.raise_for_status()
        except requests.HTTPError as e:
            print(f"[WARN] {e} -> {url_pag}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        items = soup.select("li.ui-search-layout__item, li.ui-search-layout__item--stack")
        if not items:
            items = soup.select("li.ui-search-result__wrapper")
        if not items:
            print(f"[WARN] No se hallaron items en página {page}.")
            break

        for li in items:
            try:
                articulos.append(parse_item(li))
            except Exception as exc:
                print(f"[WARN] item con error: {exc}")

        time.sleep(ESPERA_S)

    # FIX: el return ahora está FUERA del loop de páginas
    return pd.DataFrame(articulos)

# Scraping principal

In [9]:
PAGINAS = 4

# Cargar checkpoint para reanudar si el proceso se interrumpió
if os.path.exists(ruta_checkpoint):
    with open(ruta_checkpoint) as f:
        urls_procesadas = set(json.load(f))
    print(f"Reanudando: {len(urls_procesadas)} URLs ya procesadas")
else:
    urls_procesadas = set()

urls_pendientes = [u for u in urls if u not in urls_procesadas]
print(f"URLs pendientes: {len(urls_pendientes)}")

for i, url_base in enumerate(tqdm(urls_pendientes)):
    print(f"\n[{i+1}/{len(urls_pendientes)}] {url_base}")

    try:
        df_scrap = scrap_page(PAGINAS, url_base, HEADERS)
    except Exception as e:
        print(f"[ERROR] Falló scrap_page en {url_base}: {e}")
        continue

    if df_scrap is None or df_scrap.empty:
        print(f"[WARN] No se obtuvo información para {url_base}")
        urls_procesadas.add(url_base)
        continue

    print(f"Scraped: {df_scrap.shape[0]} artículos")

    # Coordenadas
    lats, lons = [], []
    for j in range(df_scrap.shape[0]):
        print(f"  Coordenadas {j+1}/{df_scrap.shape[0]}")
        lat, lon = get_lat_lon(df_scrap.loc[j, 'link'])
        lats.append(lat)
        lons.append(lon)
        time.sleep(random.randint(15, 30))

    df_scrap['latitud'] = lats   # FIX: columna 'latitud' (antes 'atitud')
    df_scrap['longitud'] = lons

    # Descripciones
    descs = []
    for j in range(df_scrap.shape[0]):
        print(f"  Descripción {j+1}/{df_scrap.shape[0]}")
        descs.append(get_descripcion(df_scrap.loc[j, 'link']))
        time.sleep(random.randint(15, 30))

    df_scrap['descripcion'] = descs

    # Metadatos de la URL
    tipo_vivienda, operacion, estado, municipio = extraer_datos_url(url_base)
    df_scrap['tipo_vivienda'] = tipo_vivienda
    df_scrap['operacion'] = operacion
    df_scrap['estado'] = estado
    df_scrap['municipio'] = municipio

    guardar_dataframe(df_scrap, ruta_archivo)

    # Guardar checkpoint
    urls_procesadas.add(url_base)
    with open(ruta_checkpoint, 'w') as f:
        json.dump(list(urls_procesadas), f)

print("\n✅ Scraping completado.")

URLs pendientes: 776


  0%|                                                   | 0/776 [00:00<?, ?it/s]


[1/776] https://inmuebles.mercadolibre.com.mx/casas/venta/distrito-federal/alvaro-obregon/
Scrapeando página 1
Scrapeando página 2
Scrapeando página 3
Scrapeando página 4
Scraped: 192 artículos
  Coordenadas 1/192
  Coordenadas 2/192
  Coordenadas 3/192
  Coordenadas 4/192
  Coordenadas 5/192
  Coordenadas 6/192
  Coordenadas 7/192
  Coordenadas 8/192
  Coordenadas 9/192
  Coordenadas 10/192
  Coordenadas 11/192
  Coordenadas 12/192
  Coordenadas 13/192
  Coordenadas 14/192
  Coordenadas 15/192
  Coordenadas 16/192
  Coordenadas 17/192
  Coordenadas 18/192
  Coordenadas 19/192
  Coordenadas 20/192
  Coordenadas 21/192
  Coordenadas 22/192
  Coordenadas 23/192
  Coordenadas 24/192
  Coordenadas 25/192
  Coordenadas 26/192
  Coordenadas 27/192
  Coordenadas 28/192
  Coordenadas 29/192
  Coordenadas 30/192
  Coordenadas 31/192
  Coordenadas 32/192
  Coordenadas 33/192
  Coordenadas 34/192
  Coordenadas 35/192
  Coordenadas 36/192
  Coordenadas 37/192
  Coordenadas 38/192
  Coordenadas 39

  0%|                                 | 1/776 [4:50:59<3758:44:20, 17459.95s/it]


[2/776] https://inmuebles.mercadolibre.com.mx/casas/renta/distrito-federal/alvaro-obregon/
Scrapeando página 1
Scrapeando página 2
Scrapeando página 3
Scrapeando página 4
Scraped: 192 artículos
  Coordenadas 1/192
  Coordenadas 2/192
  Coordenadas 3/192
  Coordenadas 4/192
  Coordenadas 5/192
  Coordenadas 6/192
  Coordenadas 7/192
  Coordenadas 8/192
  Coordenadas 9/192
  Coordenadas 10/192
  Coordenadas 11/192
  Coordenadas 12/192
  Coordenadas 13/192
  Coordenadas 14/192
  Coordenadas 15/192
  Coordenadas 16/192
  Coordenadas 17/192
  Coordenadas 18/192
  Coordenadas 19/192
  Coordenadas 20/192
  Coordenadas 21/192
  Coordenadas 22/192
  Coordenadas 23/192
  Coordenadas 24/192
  Coordenadas 25/192
  Coordenadas 26/192
  Coordenadas 27/192
  Coordenadas 28/192
  Coordenadas 29/192
  Coordenadas 30/192
  Coordenadas 31/192
  Coordenadas 32/192
  Coordenadas 33/192
  Coordenadas 34/192
  Coordenadas 35/192
  Coordenadas 36/192
  Coordenadas 37/192
  Coordenadas 38/192
  Coordenadas 39

  0%|                                 | 2/776 [9:49:57<3814:17:58, 17740.93s/it]


[3/776] https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/alvaro-obregon/
Scrapeando página 1
Scrapeando página 2
Scrapeando página 3
Scrapeando página 4
Scraped: 192 artículos
  Coordenadas 1/192
  Coordenadas 2/192
  Coordenadas 3/192
  Coordenadas 4/192
  Coordenadas 5/192
  Coordenadas 6/192
  Coordenadas 7/192
  Coordenadas 8/192
  Coordenadas 9/192
  Coordenadas 10/192
  Coordenadas 11/192
  Coordenadas 12/192
  Coordenadas 13/192
  Coordenadas 14/192
  Coordenadas 15/192
  Coordenadas 16/192
  Coordenadas 17/192
  Coordenadas 18/192
  Coordenadas 19/192
  Coordenadas 20/192
  Coordenadas 21/192
  Coordenadas 22/192
  Coordenadas 23/192
  Coordenadas 24/192
  Coordenadas 25/192
  Coordenadas 26/192
  Coordenadas 27/192
  Coordenadas 28/192
  Coordenadas 29/192
  Coordenadas 30/192
  Coordenadas 31/192
  Coordenadas 32/192
  Coordenadas 33/192
  Coordenadas 34/192
  Coordenadas 35/192
  Coordenadas 36/192
  Coordenadas 37/192
  Coordenadas 38/192
  Coorde

  0%|                                | 2/776 [11:22:03<4399:16:48, 20461.77s/it]


KeyboardInterrupt: 